In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.use_database("bootcamp")
session.use_schema("atp524")  # Replace with your schema.

In [ ]:
import importlib.util
import sys
from pathlib import Path

# The scripts have numbered filenames (e.g. 03.ingest_sec_ground_truth.py), which
# `import` cannot load, so load them by path under their original module names.
cwd = Path.cwd().resolve()
python_dir = next(
    d for d in (cwd, cwd / "python", *cwd.parents)
    if (d / "03.ingest_sec_ground_truth.py").exists()
)
if str(python_dir) not in sys.path:
    sys.path.insert(0, str(python_dir))  # for their own imports, e.g. sec_facts

def load_module(name, filename):
    spec = importlib.util.spec_from_file_location(name, python_dir / filename)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

ingest_sec_ground_truth = load_module("ingest_sec_ground_truth", "03.ingest_sec_ground_truth.py")
ingest_annual_report_pdfs = load_module("ingest_annual_report_pdfs", "02.ingest_annual_report_pdfs.py")

## purpose: contact string sent to SEC with every download, and the SEC asks automated requests to include one. the below is a bit NULL and VOID since we are not runnnig the ingestion in this notebook as well
## as we set these in other scripts so keeping/removing changes nothing. but in theory, could set them in once place here and run ingestion here.
## the below mimics what we have in other scripts (ingest_annual_report_pdfs and ingest_sec_ground_truth) so I have copied the same but in theory can set to what you want
ingest_sec_ground_truth.SEC_USER_AGENT = "FinancialDocumentsCourse zach@dataexpert.io"
##"AnnualReportsLab your.real.contact@example.com"
ingest_annual_report_pdfs.SEC_USER_AGENT = "FinancialDocumentsCourse zach@dataexpert.io"

In [ ]:
%%sql -r report_setting
-- Choose what to process: the ONLY place the ticker and year range are set.
-- 02 (PDFs) and 03 (SEC facts) ingest this scope, and every SQL step from 04 on
-- processes every report in it, so no ingested year is silently skipped.
-- Check coverage with pipeline_coverage (06.check_readiness.sql).
-- To process a single report, set first_year = last_year.
USE WAREHOUSE compute_wh;
MERGE INTO bootcamp.atp524.lab_settings t
USING (SELECT 'AAPL' AS ticker, 2023 AS first_year, 2024 AS last_year) s ON TRUE
WHEN MATCHED THEN UPDATE SET ticker = s.ticker, first_year = s.first_year,
                             last_year = s.last_year, updated_at = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN INSERT (ticker, first_year, last_year)
                      VALUES (s.ticker, s.first_year, s.last_year);

SELECT ticker, first_year, last_year, updated_at FROM bootcamp.atp524.lab_settings;